In [10]:
import os
import random # 仅用于shuffle数据，不再固定种子
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

# ================= 配置区域 =================
DEVICE = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
DATASET_ROOT = "./dataset_tri_RAD_Tensor"
SIM_TRAIN_DIR = os.path.join(DATASET_ROOT, "sim_train")
SIM_VAL_DIR = os.path.join(DATASET_ROOT, "sim_val")
REAL_TEST_DIR = os.path.join(DATASET_ROOT, "real_test")
RESULTS_DIR = "./training_results_breakthrough"

CLASS_NAMES = [
    "BPSK", "QPSK", "8PSK", "OQPSK", "PI4DQPSK", 
    "16QAM", "32QAM", "64QAM", 
    "2ASK", "16APSK", "32APSK", "FSK2"
]
NUM_CLASSES = len(CLASS_NAMES)
BATCH_SIZE = 128
# [保持] 0.001 配合余弦退火是最佳组合
LEARNING_RATE = 0.001 
# [增加] 给模型更多时间去抠细节
NUM_EPOCHS = 30 

os.makedirs(RESULTS_DIR, exist_ok=True)
BEST_MODEL_PATH = os.path.join(RESULTS_DIR, "best_model.pth")

# [关键修改 1] 彻底删除 seed_everything
# 让每次初始化的权重都随机，增加“撞大运”的几率

# ================= 1. 数据集 (保持纯净) =================
class TensorDataset(Dataset):
    def __init__(self, root_dir):
        self.data = []
        self.labels = []
        if not os.path.exists(root_dir): return
        
        # 即使不固定种子，数据加载的顺序随机性也很重要
        files = []
        for cls_name in CLASS_NAMES:
            d = Path(root_dir) / cls_name
            if d.exists():
                files.extend([(p, CLASS_NAMES.index(cls_name)) for p in d.glob("*.pt")])
        random.shuffle(files) 
        
        for p, label in files:
            try:
                self.data.append(torch.load(p))
                self.labels.append(label)
            except: pass

    def __len__(self): return len(self.data)
    def __getitem__(self, idx): return self.data[idx], self.labels[idx]

# ================= 2. 模型升级：MicroResNet =================
# 这是一个比 TinyCNN 强，但比标准 ResNet18 轻量得多的模型
# 专门为了解决“卡在90%”而设计
class BasicBlock(nn.Module):
    def __init__(self, in_planes, planes, stride=1):
        super(BasicBlock, self).__init__()
        self.conv1 = nn.Conv2d(in_planes, planes, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(planes)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(planes, planes, kernel_size=3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(planes)
        
        self.shortcut = nn.Sequential()
        if stride != 1 or in_planes != planes:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_planes, planes, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(planes)
            )

    def forward(self, x):
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += self.shortcut(x) # 残差连接：防止退化，保证能学到东西
        out = self.relu(out)
        return out

class MicroResNet(nn.Module):
    def __init__(self, num_classes):
        super(MicroResNet, self).__init__()
        # 初始层
        self.in_planes = 64
        self.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        self.relu = nn.ReLU(inplace=True)
        
        # 三层残差块 (逐步加宽)
        # 64 -> 64
        self.layer1 = self._make_layer(64, 2, stride=1) 
        # 64 -> 128 (尺寸减半)
        self.layer2 = self._make_layer(128, 2, stride=2)
        # 128 -> 256 (尺寸再减半)
        self.layer3 = self._make_layer(256, 2, stride=2)
        
        # 使用 GAP 取代 Flatten，极大减少参数量，防止过拟合
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(256, num_classes)
        
        self._initialize_weights()

    def _make_layer(self, planes, num_blocks, stride):
        strides = [stride] + [1]*(num_blocks-1)
        layers = []
        for stride in strides:
            layers.append(BasicBlock(self.in_planes, planes, stride))
            self.in_planes = planes
        return nn.Sequential(*layers)

    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)

    def forward(self, x):
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.layer1(out)
        out = self.layer2(out)
        out = self.layer3(out)
        out = self.avgpool(out)
        out = out.flatten(1)
        out = self.fc(out)
        return out

# ================= 3. 主逻辑 =================
def main():
    # [关键修改 1] 不设置随机种子
    print("Random Seed: NOT SET (Letting luck play a role)")

    train_ds = TensorDataset(SIM_TRAIN_DIR)
    val_ds = TensorDataset(SIM_VAL_DIR)
    real_ds = TensorDataset(REAL_TEST_DIR)
    
    # num_workers=4 可以加速数据加载，如果报错请改回 0
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    
    # 使用 MicroResNet
    model = MicroResNet(NUM_CLASSES).to(DEVICE)
    
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
    criterion = nn.CrossEntropyLoss()
    
    # 保持余弦退火，因为它真的很稳
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)
    
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
    best_val_acc = 0.0
    
    print(f"Starting Training for {NUM_EPOCHS} epochs...")
    for epoch in range(NUM_EPOCHS):
        model.train()
        running_loss, correct, total = 0.0, 0, 0
        
        for imgs, labels in train_loader: 
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            loss.backward()
            
            # [关键] 必须保留梯度裁剪，防止不稳定
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            
            optimizer.step()
            
            running_loss += loss.item() * imgs.size(0)
            _, preds = outputs.max(1)
            correct += preds.eq(labels).sum().item()
            total += labels.size(0)
            
        epoch_loss = running_loss / total
        epoch_acc = correct / total
        
        # Validation
        model.eval()
        val_loss_sum, val_correct, val_total = 0.0, 0, 0
        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
                outputs = model(imgs)
                val_loss_sum += criterion(outputs, labels).item() * imgs.size(0)
                _, preds = outputs.max(1)
                val_correct += preds.eq(labels).sum().item()
                val_total += labels.size(0)
        
        val_loss = val_loss_sum / val_total
        val_acc = val_correct / val_total
        
        scheduler.step()
        
        history['train_loss'].append(epoch_loss)
        history['train_acc'].append(epoch_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        
        save_msg = ""
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), BEST_MODEL_PATH)
            save_msg = " [New Best!]"
            
        # 打印学习率看衰减情况
        current_lr = optimizer.param_groups[0]['lr']
        print(f"Epoch {epoch+1:02d} | T_Acc: {epoch_acc:.4f} | V_Acc: {val_acc:.4f} | LR: {current_lr:.6f}{save_msg}")

    # 绘图
    epochs = range(1, len(history['train_acc']) + 1)
    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1); plt.plot(epochs, history['train_loss'], label='Train'); plt.plot(epochs, history['val_loss'], label='Val'); plt.title('Loss')
    plt.subplot(1, 2, 2); plt.plot(epochs, history['train_acc'], label='Train'); plt.plot(epochs, history['val_acc'], label='Val'); plt.title('Acc')
    plt.savefig(os.path.join(RESULTS_DIR, "curves.png"))
    
    # 真实数据测试
    if len(real_ds) > 0:
        model.load_state_dict(torch.load(BEST_MODEL_PATH))
        model.eval()
        real_loader = DataLoader(real_ds, batch_size=BATCH_SIZE)
        y_true, y_pred = [], []
        with torch.no_grad():
            for imgs, labels in real_loader:
                imgs = imgs.to(DEVICE)
                _, preds = model(imgs).max(1)
                y_true.extend(labels.cpu().numpy())
                y_pred.extend(preds.cpu().numpy())
        acc = np.mean(np.array(y_true) == np.array(y_pred))
        print(f"\nReal Data Accuracy: {acc:.4f}")
        
        existing_idx = sorted(list(set(y_true)))
        existing_names = [CLASS_NAMES[i] for i in existing_idx]
        plot_confusion_matrix(y_true, y_pred, f"Real Data (Acc: {acc:.2%})", 
                             os.path.join(RESULTS_DIR, "real_cm.png"), target_names=existing_names, labels=existing_idx)

def plot_confusion_matrix(y_true, y_pred, title, save_path, target_names=None, labels=None):
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=target_names, yticklabels=target_names)
    plt.title(title); plt.tight_layout(); plt.savefig(save_path); plt.close()

if __name__ == "__main__":
    main()

Random Seed: NOT SET (Letting luck play a role)
Starting Training for 30 epochs...
Epoch 01 | T_Acc: 0.6714 | V_Acc: 0.6578 | LR: 0.000997 [New Best!]
Epoch 02 | T_Acc: 0.8315 | V_Acc: 0.5767 | LR: 0.000989
Epoch 03 | T_Acc: 0.8439 | V_Acc: 0.5861 | LR: 0.000976
Epoch 04 | T_Acc: 0.8639 | V_Acc: 0.7285 | LR: 0.000957 [New Best!]
Epoch 05 | T_Acc: 0.8735 | V_Acc: 0.8571 | LR: 0.000933 [New Best!]
Epoch 06 | T_Acc: 0.8838 | V_Acc: 0.8030 | LR: 0.000905
Epoch 07 | T_Acc: 0.9015 | V_Acc: 0.7003 | LR: 0.000872
Epoch 08 | T_Acc: 0.9034 | V_Acc: 0.6794 | LR: 0.000835
Epoch 09 | T_Acc: 0.9165 | V_Acc: 0.8085 | LR: 0.000794
Epoch 10 | T_Acc: 0.9165 | V_Acc: 0.8129 | LR: 0.000750
Epoch 11 | T_Acc: 0.9246 | V_Acc: 0.8951 | LR: 0.000703 [New Best!]
Epoch 12 | T_Acc: 0.9301 | V_Acc: 0.8135 | LR: 0.000655
Epoch 13 | T_Acc: 0.9372 | V_Acc: 0.8538 | LR: 0.000604


KeyboardInterrupt: 